# SOTA LLM Embedding 2

**Goal:** explore text and image embeddings using open-source models, Gemini API, and CLIP — then compare semantic search vs keyword search.

**Flow:** API setup → Open-source text embeddings → Gemini text embeddings → Gemini image embedding → CLIP image features → Semantic similarity search → Keyword search baseline

In [2]:
import warnings
warnings.filterwarnings("ignore", message="IProgress not found.*")
from sentence_transformers import SentenceTransformer
print('All imports successful')

All imports successful


## Step 1: Import and configure API access

- Import `SentenceTransformer` for open-source embeddings.
- Map environment variables to the standard names SDKs expect (`GOOGLE_API_KEY`, `OPENAI_API_KEY`, etc.).
- Display placeholder-safe key status — never print real credentials.

**Why map env vars?** SDKs like LangChain and google-genai look for specific variable names. Mapping once avoids hardcoding keys throughout the notebook.

In [4]:
import os
os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.getenv("HUGGINGFACEHUB_API_TOKEN") or os.getenv("HF", "")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY") or os.getenv("OPENAI", "")
os.environ["GOOGLE_API_KEY"] = (
    os.getenv("GOOGLE_API_KEY")
    or os.getenv("GEMINI_API_KEY")
    or os.getenv("GEMINI", "")
)

print("API keys mapped to standard environment variable names")

API keys mapped to standard environment variable names


In [87]:
display_hf_key = "<HF_TOKEN_PLACEHOLDER>" if (os.getenv("HF") or os.getenv("HUGGINGFACEHUB_API_TOKEN")) else None
display_openai_key = "<OPENAI_API_KEY_PLACEHOLDER>" if (os.getenv("OPENAI") or os.getenv("OPENAI_API_KEY")) else None
display_gemini_key = "<GOOGLE_API_KEY_PLACEHOLDER>" if (os.getenv("GEMINI") or os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")) else None

display_hf_key, display_openai_key, display_gemini_key

('<HF_TOKEN_PLACEHOLDER>',
 '<OPENAI_API_KEY_PLACEHOLDER>',
 '<GOOGLE_API_KEY_PLACEHOLDER>')

## Step 2: Open-source text embeddings (Sentence Transformers)

- Fallback loop tries three Hugging Face models until one loads.
- `.encode()` converts any text (word, sentence, paragraph) into a fixed-size dense vector.
- Shape is model-dependent: 384-dim for MiniLM, 768 for larger models.

**Why open-source first?** No API key needed after download. Fast local inference. Full control over the model.

**Alternative:** use `transformers` directly with manual tokenization and pooling for custom embedding behavior.

In [6]:
model_candidates = [
    'sentence-transformers/all-MiniLM-L6-v2',
    'sentence-transformers/paraphrase-MiniLM-L6-v2',
    'BAAI/bge-small-en-v1.5',
]

open_source_embedding_model = None
selected_model_name = None
load_errors = {}

for model_name in model_candidates:
    try:
        open_source_embedding_model = SentenceTransformer(model_name)
        selected_model_name = model_name
        break
    except Exception as exc:
        load_errors[model_name] = str(exc)

if open_source_embedding_model is None:
    raise RuntimeError(
        'No embedding model could be loaded. If SSL blocks Hugging Face access, '
        'download one model manually and pass its local folder path to SentenceTransformer(...).\n'
        f'Errors: {load_errors}'
    )

print(f'Embedding model loaded successfully: {selected_model_name}')

Embedding model loaded successfully: sentence-transformers/all-MiniLM-L6-v2


In [7]:
text="hello"

In [8]:
embedding = open_source_embedding_model.encode(text)
embedding[:10]

array([-0.0627718 ,  0.05495879,  0.05216482,  0.08578998, -0.08274891,
       -0.07457295,  0.06855472,  0.01839637, -0.08201133, -0.03738481],
      dtype=float32)

In [9]:
len(embedding)

384

In [10]:
embedding.shape

(384,)

In [11]:
sentence = 'Exercise is good for health'

In [12]:
embedding_sent = open_source_embedding_model.encode(sentence)
embedding[:15]

array([-0.0627718 ,  0.05495879,  0.05216482,  0.08578998, -0.08274891,
       -0.07457295,  0.06855472,  0.01839637, -0.08201133, -0.03738481,
        0.01212489,  0.00351838, -0.00413433, -0.04378442,  0.02180735],
      dtype=float32)

In [13]:
embedding.shape

(384,)

In [14]:
len(embedding)

384

In [15]:
paragraph = "Data centres are the backbone of modern digital life, housing servers that store, process, and deliver information for websites, apps, and cloud platforms. They require reliable power, advanced cooling, strong cybersecurity, and constant monitoring to maintain uptime. As demand for AI and streaming grows, operators are improving efficiency through virtualization, renewable energy, and smarter hardware design, making data centres more scalable, resilient, and environmentally responsible for future global digital services."

## Step 3: Gemini text embeddings

- `GoogleGenerativeAIEmbeddings` from LangChain wraps the Gemini embedding API.
- `embed_query()` returns a single embedding as a Python list (not NumPy).
- Wrap with `np.array()` to use `.shape`.

**Why Gemini?** API-based models don't need local GPU resources. Good for production-scale embedding with managed infrastructure.

**Alternative:** OpenAI `text-embedding-ada-002`, Cohere `embed-v3`, or any hosted embedding API.

In [17]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os

google_api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")

if not google_api_key:
    raise ValueError("Set GOOGLE_API_KEY or GEMINI_API_KEY before creating the Gemini embedding model.")

google_embedding_model = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=google_api_key,
)

print("Gemini text embedding model initialized successfully")

Gemini text embedding model initialized successfully


In [18]:
embedding_gogl=google_embedding_model.embed_query("where is bihar located on earth?")
embedding_gogl[:30]

[-0.011071488,
 -0.008674031,
 -0.013936672,
 -0.06700222,
 0.009862963,
 0.004321306,
 0.011519839,
 -0.0025798338,
 0.0047839414,
 0.011180122,
 0.006495429,
 -0.015340364,
 -0.011399111,
 0.052388657,
 0.12558861,
 -0.006592585,
 0.0014810686,
 -0.010916202,
 0.012178439,
 -0.005350754,
 -0.028936643,
 0.015953898,
 0.014689316,
 -0.04296307,
 0.0003587547,
 -0.0065929797,
 -0.008686692,
 -0.010928572,
 0.012311199,
 0.015232021]

In [19]:
import numpy as np
np.array(embedding_gogl).shape 

(3072,)

In [20]:
len(embedding_gogl)

3072

## Step 4: Gemini image embedding

- Load the local image file as raw bytes.
- Wrap in `types.Part.from_bytes(...)` with the correct MIME type.
- Send to `gemini-embedding-2` via `client.models.embed_content()`.
- Extract the vector from `result.embeddings[0].values`.

**Why image embeddings?** Enables multimodal search — find images by text query or compare images by semantic content.

**Alternative:** use CLIP (shown next) for local image embedding without API dependency.

In [21]:
from google import genai
from google.genai import types

client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

image_path = "../ai.jpg"
with open(image_path, "rb") as f:
    image_bytes = f.read()

result = client.models.embed_content(
    model="gemini-embedding-2",
    contents=[types.Part.from_bytes(data=image_bytes, mime_type="image/jpeg")],
)

image_embedding = result.embeddings[0].values
print(f"Image embedding generated successfully, embedding length: {len(image_embedding)}")
#we can use clipmodel /clipprocessor for image embedding as well

Image embedding generated successfully, embedding length: 3072


In [22]:
image_embedding[:30]

[-0.009680204,
 0.010199237,
 -0.031110514,
 0.01868216,
 0.014802262,
 -0.0037867085,
 -0.01945133,
 0.0013252717,
 0.0015744586,
 -0.035983548,
 0.00235209,
 0.0115877325,
 -0.005151765,
 -0.0086993,
 -0.0055219526,
 -0.014986239,
 0.0070156367,
 0.009140471,
 0.032231197,
 -0.0057983925,
 0.0040315,
 0.0047016423,
 -0.016285866,
 0.034740042,
 -0.0016579822,
 -0.017667409,
 -0.005341294,
 0.003376664,
 -0.016731063,
 0.057392746]

## Step 5: CLIP image embeddings

- `CLIPProcessor` preprocesses the image into tensor inputs the model expects.
- `CLIPModel.get_image_features()` extracts the image embedding.
- `torch.no_grad()` disables gradient computation — inference only.
- Output shape: `(1, 512)` — the 1 is the batch dimension.

**Why CLIP?** It maps images and text into the same embedding space. You can compare an image embedding with a text embedding directly — powerful for multimodal retrieval.

**Alternative:** SigLIP, BLIP-2, or other vision-language models for more recent architectures.

In [23]:
from transformers import CLIPProcessor, CLIPModel
#VIT and CLIP 

In [24]:
from PIL import Image

In [25]:
image = Image.open(image_path).convert("RGB")

In [26]:
import os
import warnings

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
warnings.filterwarnings("ignore", message="Using a slow image processor.*")

processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32",
    use_fast=True,
)

In [27]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")

In [28]:
image_inputs = processor(images=image, return_tensors="pt", padding=True)
image_inputs

{'pixel_values': tensor([[[[-0.6974, -0.7412, -0.4930,  ...,  1.5216,  1.5362,  1.3610],
          [-0.3324, -0.1572, -0.1718,  ...,  1.3026,  1.1858,  1.1420],
          [-0.2302, -0.0696,  0.0909,  ...,  1.3756,  1.2296,  0.9668],
          ...,
          [-0.4346, -0.4200, -0.4200,  ..., -0.2886, -0.2886, -0.3032],
          [-0.3908, -0.4054, -0.5076,  ..., -0.3178, -0.3032, -0.2886],
          [-0.6828, -0.6536, -0.6536,  ..., -0.2886, -0.3470, -0.2302]],

         [[-0.7766, -0.7916, -0.6865,  ...,  0.6792,  0.7842,  0.1839],
          [-0.6265, -0.5665, -0.5515,  ...,  0.1689,  0.0188,  0.0789],
          [-0.5965, -0.5365, -0.4614,  ...,  0.3640,  0.3790, -0.0262],
          ...,
          [ 0.6942,  0.6792,  0.6642,  ...,  0.2439,  0.2439,  0.2439],
          [ 0.7242,  0.6942,  0.6341,  ...,  0.2289,  0.2289,  0.2439],
          [ 0.5141,  0.5441,  0.5441,  ...,  0.2740,  0.1989,  0.3040]],

         [[-0.6839, -0.6981, -0.6412,  ...,  0.2688,  0.3684, -0.2289],
          [-0

In [29]:
import torch

with torch.no_grad():
    image_features = model.get_image_features(**image_inputs)

image_features


tensor([[-6.2313e-02,  8.4444e-02, -7.4536e-02, -1.4438e-01,  3.2303e-01,
         -6.2261e-03,  1.7565e-01,  4.9803e-01, -1.1739e-01, -3.7875e-01,
          2.5473e-01, -2.1208e-02, -2.7136e-01, -2.1172e-02, -3.9018e-02,
         -9.0900e-02, -4.5676e-01,  5.9715e-01, -1.6957e-01,  5.6386e-02,
          2.4201e-01,  3.6250e-01,  5.3526e-01,  3.3031e-01, -3.1658e-01,
         -3.1753e-01,  2.3999e-01, -5.9463e-02,  3.5932e-01, -4.8186e-01,
         -2.8675e-02, -8.7247e-02, -2.1608e-01,  6.9960e-02, -5.3482e-01,
         -1.4713e-01,  9.2075e-03,  1.8662e-01, -2.7959e-01, -1.5275e+00,
         -6.0090e-01, -6.8739e-02,  4.0092e-01, -2.4058e-02, -4.1878e-02,
          7.3976e-01,  2.1463e-01,  3.1386e-01, -2.7486e-02,  5.7937e-01,
          3.1689e-01, -3.5185e-02,  3.0558e-01,  1.1435e-01, -2.9639e-01,
         -2.9733e-01,  2.6246e-01,  5.2935e-01, -2.8146e-01, -3.7945e-01,
          6.1173e-02, -5.8513e-02, -7.8159e-02,  8.2522e-02, -1.9679e-01,
         -4.0541e-01, -9.1026e-02,  8.

In [30]:
len(image_features) #1
len(image_features[0])

512

## Step 6: Semantic search using embeddings

- Embed the query and all documents with Gemini.
- Compare each document to the query using three metrics:
  - **Cosine similarity:** direction-based, [-1, 1]. Most common for search.
  - **Dot product:** includes magnitude, (-∞, +∞). Use when vectors are pre-normalized.
  - **Euclidean distance:** straight-line distance, [0, +∞). Lower = more similar.
- Results stored as a list of dicts for easy inspection.

**Why three metrics?** Different tasks and models favor different metrics. Cosine is the default baseline; dot product is faster if vectors are already normalized; Euclidean captures absolute distance.

In [ ]:
def dot_product(a, b):
    return np.dot(a, b)

In [50]:
def cosine_similarity(a, b):
    return np.dot(a,b)/ (np.linalg.norm(a)* np.linalg.norm(b))

In [51]:
def euclidean_distance(a,b):
    a=np.array(a)
    b=np.array(b)
    return np.linalg.norm(a - b)

In [52]:
query = " How water is used in data centres?"

In [62]:
document = [
    "Water is used in many data centres for cooling systems, especially in chilled water loops and cooling towers that remove heat from servers.",
    "Some modern data centres reduce water usage by using air cooling, liquid immersion cooling, or recycled water systems to improve sustainability.",
    "Data centre operators monitor water usage effectiveness (WUE) to measure how efficiently a facility uses water for cooling and operations.",
    "Servers in data centres generate significant heat, so cooling infrastructure is essential to maintain safe operating temperatures and prevent hardware failure.",
    "Renewable energy and efficient hardware design help data centres reduce both electricity consumption and environmental impact.",
    "A mango tree grows well in warm climates and produces sweet tropical fruit during the harvest season.",
    "Football teams train daily to improve passing, defense, and match fitness before major tournaments."
]

In [63]:
query_embedding = google_embedding_model.embed_query(query)


In [64]:
doc_embedding= google_embedding_model.embed_documents(document)

In [65]:
len(doc_embedding), len(doc_embedding[0])

(7, 3072)

In [66]:
res = []

for doc, doc_emb in zip(document, doc_embedding):
    res.append({
        "query": query,
        "document": doc,
        "cosine_similarity": cosine_similarity(query_embedding, doc_emb),
        "dot_product": dot_product(query_embedding, doc_emb),
        "euclidean_distance": euclidean_distance(query_embedding, doc_emb),
    })

res

[{'query': ' How water is used in data centres?',
  'document': 'Water is used in many data centres for cooling systems, especially in chilled water loops and cooling towers that remove heat from servers.',
  'cosine_similarity': 0.8298547446000345,
  'dot_product': 0.8298547572935759,
  'euclidean_distance': 0.5833442517116716},
 {'query': ' How water is used in data centres?',
  'document': 'Some modern data centres reduce water usage by using air cooling, liquid immersion cooling, or recycled water systems to improve sustainability.',
  'cosine_similarity': 0.8157680198474242,
  'dot_product': 0.8157680836561213,
  'euclidean_distance': 0.6070123467657849},
 {'query': ' How water is used in data centres?',
  'document': 'Data centre operators monitor water usage effectiveness (WUE) to measure how efficiently a facility uses water for cooling and operations.',
  'cosine_similarity': 0.791034384519601,
  'dot_product': 0.7910343933498789,
  'euclidean_distance': 0.6464760131869892},
 

## Step 7: Keyword search baseline

- Tokenize query and documents by splitting on whitespace after lowercasing.
- Use `Counter` to count word frequencies in each document.
- Score = sum of query word counts found in the document.
- Wrapped into reusable functions for clean comparison.

**Why include keyword search?** It's the simplest retrieval baseline. Comparing it against semantic search shows where embeddings capture meaning that exact token matching misses (e.g., the mango/football docs score near-zero in both, but semantically related docs rank much higher with embeddings).

**Alternative:** BM25 for a stronger lexical baseline that accounts for term frequency saturation and document length normalization.

In [67]:
query

' How water is used in data centres?'

In [68]:
from collections import Counter

In [79]:
quer_words=query.lower().replace("?", "").split()
quer_words

['how', 'water', 'is', 'used', 'in', 'data', 'centres']

In [80]:
doc_words=document[0].lower().replace(".", "").split()
doc_words

['water',
 'is',
 'used',
 'in',
 'many',
 'data',
 'centres',
 'for',
 'cooling',
 'systems,',
 'especially',
 'in',
 'chilled',
 'water',
 'loops',
 'and',
 'cooling',
 'towers',
 'that',
 'remove',
 'heat',
 'from',
 'servers']

In [81]:
doc_counter = Counter(doc_words)
doc_counter

Counter({'water': 2,
         'in': 2,
         'cooling': 2,
         'is': 1,
         'used': 1,
         'many': 1,
         'data': 1,
         'centres': 1,
         'for': 1,
         'systems,': 1,
         'especially': 1,
         'chilled': 1,
         'loops': 1,
         'and': 1,
         'towers': 1,
         'that': 1,
         'remove': 1,
         'heat': 1,
         'from': 1,
         'servers': 1})

In [82]:
for word in quer_words:
    print(f"Word: {word}, Count in document: {doc_counter.get(word, 0)}")


Word: how, Count in document: 0
Word: water, Count in document: 2
Word: is, Count in document: 1
Word: used, Count in document: 1
Word: in, Count in document: 2
Word: data, Count in document: 1
Word: centres, Count in document: 1


In [84]:
score=sum([doc_counter.get(word, 0) for word in quer_words])
score

8

In [86]:
def tokenize_for_keyword_search(text):
    return text.lower().replace(".", "").replace(",", "").replace("?", "").split()


def keyword_score(query_words, doc_text):
    doc_words = tokenize_for_keyword_search(doc_text)
    doc_counter = Counter(doc_words)
    return sum(doc_counter.get(word, 0) for word in query_words)


def keyword_search_all_documents(query_text, documents):
    query_words = tokenize_for_keyword_search(query_text)
    results = []

    for idx, doc in enumerate(documents, start=1):
        results.append({
            "doc_id": idx,
            "document": doc,
            "keyword_score": keyword_score(query_words, doc),
        })

    return sorted(results, key=lambda x: x["keyword_score"], reverse=True)


keyword_results = keyword_search_all_documents(query, document)
keyword_results

[{'doc_id': 1,
  'document': 'Water is used in many data centres for cooling systems, especially in chilled water loops and cooling towers that remove heat from servers.',
  'keyword_score': 8},
 {'doc_id': 2,
  'document': 'Some modern data centres reduce water usage by using air cooling, liquid immersion cooling, or recycled water systems to improve sustainability.',
  'keyword_score': 4},
 {'doc_id': 3,
  'document': 'Data centre operators monitor water usage effectiveness (WUE) to measure how efficiently a facility uses water for cooling and operations.',
  'keyword_score': 4},
 {'doc_id': 4,
  'document': 'Servers in data centres generate significant heat, so cooling infrastructure is essential to maintain safe operating temperatures and prevent hardware failure.',
  'keyword_score': 4},
 {'doc_id': 5,
  'document': 'Renewable energy and efficient hardware design help data centres reduce both electricity consumption and environmental impact.',
  'keyword_score': 2},
 {'doc_id': 6,

## Revision notes

- **Open-source models** (SentenceTransformer) run locally after download — no recurring API costs.
- **Gemini embeddings** support both text and images via API — good for production but needs network + API key.
- **CLIP** maps images and text into a shared 512-dim space — enables cross-modal search.
- Gemini output is a Python list — wrap in `np.array()` for `.shape`.
- CLIP output includes a batch dimension: `(1, 512)` not `(512,)`.
- **Cosine similarity** is the default semantic search metric. Dot product is equivalent when vectors are normalized.
- **Euclidean distance** is inversely related — lower = more similar. Sort ascending.
- **Keyword search** counts exact token overlap — fast but misses paraphrases and synonymy.
- Semantic search captures meaning that keyword search cannot, but is computationally heavier.
- Query embedding should be computed **once** outside any loop.

**Key takeaway:** semantic search finds related meaning even when different words are used. Keyword search only matches what's literally there.

1. **"Why do you need different embedding models for text and images?"**
   Text and image data have fundamentally different structures. Specialized models learn the right representations for each modality. CLIP is special because it trains both in a shared space.

2. **"What's the difference between `embed_query()` and `embed_documents()`?"**
   Some models apply different prefixes or processing for queries vs documents (e.g., asymmetric retrieval). `embed_query` is optimized for the search query; `embed_documents` for the corpus.

3. **"Why does cosine similarity ignore vector magnitude?"**
   It normalizes by the product of norms, measuring only directional alignment. Two vectors pointing the same way score 1.0 regardless of length. This makes it robust to document length differences.

4. **"If you normalize all embeddings to unit length, what happens to dot product?"**
   It becomes identical to cosine similarity. Many search systems pre-normalize embeddings so they can use faster dot-product operations.

5. **"Why is keyword search still used in production alongside embeddings?"**
   Speed and interpretability. Keyword search (especially BM25) is fast, explainable, and handles exact-match requirements. Many production systems use hybrid retrieval: keyword + semantic.

6. **"How would you scale semantic search to millions of documents?"**
   Use a vector database (FAISS, Pinecone, Qdrant, Weaviate). They build approximate nearest-neighbor indexes for sub-millisecond search over millions of vectors.

7. **"Can CLIP embeddings be used for text-to-image search?"**
   Yes — encode the text query and all images in CLIP's shared space, then rank images by cosine similarity to the text embedding. That's the core CLIP use case.

8. **"What happens if two documents are semantically similar but factually contradictory?"**
   Embedding models often place them close together. `"The earth is flat"` and `"The earth is round"` may have high cosine similarity because the model encodes topic, not truth. This is a known limitation.

1. **"Your semantic search ranked all data-centre docs highly. How would you distinguish the 'best' answer from 'related' ones?"**
   Use a cross-encoder re-ranker. Bi-encoders (what we used) are fast but approximate. A cross-encoder takes the query-document pair together and produces a more precise relevance score — but it's slower.

2. **"Keyword search gave the mango doc a score of 0. Could there be a case where keyword search beats semantic search?"**
   Yes — for exact-match requirements like product SKUs, error codes, or proper nouns. Semantic search might match similar-sounding but wrong items. Hybrid retrieval handles both cases.

3. **"You used three similarity metrics. In production, which one would you pick?"**
   Cosine similarity is the safest default. If embeddings are pre-normalized (many models do this), dot product is equivalent and faster. Euclidean distance is useful for clustering but less common in retrieval.

4. **"How would you combine semantic and keyword search results?"**
   Reciprocal Rank Fusion (RRF) is the simplest approach — merge ranked lists by summing inverse ranks. More advanced: learn a weighted combination using a small labeled dataset.

5. **"The CLIP image embedding is 512-dim and Gemini's is different. Can you compare them?"**
   Not directly. Different models, different vector spaces. To compare, you'd need to use the same model for both modalities, or project into a shared space with learned mappings.